# Day 053 Project: Full-Stack AI App

## What You're Building

The two halves finally meet. A **FastAPI backend** (`backend.py`, with CORS) and a **Streamlit frontend** (`frontend.py`) that calls it over HTTP through your `AIAppClient`. Run the backend, run the frontend, and the browser UI talks to the API which talks to Ollama. Two files, one app.

## Project Requirements

1. Use the provided `build_api`, `add_cors`, and `AIAppClient`.
2. Build an in-process backend with `TestClient(build_api())`, wrap it in an `AIAppClient`, and exercise `health`, `chat`, `templates`, `render`.
3. Call `write_full_stack('.')` to generate `backend.py` + `frontend.py`.
4. Run `_run_project_checks()` to verify both files.
5. Then, in **two terminals**:
   `uvicorn backend:app --reload`  and  `streamlit run frontend.py`.

## Bonus Challenges

- Add a sidebar dropdown in `frontend.py` that calls `api.templates()` and lets the user run a template via `api.render(name, topic)`.
- Show the backend health as a coloured badge that refreshes each rerun.
- Add a `timeout` to the httpx client and surface a friendly message when the backend is slow.

## Provided: Backend + Client Layer + AIAppClient

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from starlette.testclient import TestClient
import httpx
import ollama


# ---- The AI backend (built on Day 52 — provided here) ----
class ChatRequest(BaseModel):
    message: str = Field(min_length=1, description='User message for the model')
    temperature: float = Field(default=0.7, ge=0.0, le=1.0)


class ChatResponse(BaseModel):
    reply: str
    model: str


class HealthResponse(BaseModel):
    status: str
    model: str


PROMPT_TEMPLATES = {
    'summary':  'Summarize the following topic in two sentences: {topic}',
    'explain':  'Explain {topic} to a complete beginner.',
    'critique': 'List three criticisms of {topic}.',
}


def run_model(model: str, prompt: str, temperature: float = 0.7) -> str:
    resp = ollama.chat(
        model=model,
        messages=[{'role': 'user', 'content': prompt}],
        options={'temperature': temperature},
    )
    return resp['message']['content'].strip()


def build_api(model: str = 'llama3.2') -> FastAPI:
    app = FastAPI(title='AI API', version='1.0.0')

    @app.get('/health', response_model=HealthResponse)
    def health():
        return HealthResponse(status='ok', model=model)

    @app.get('/templates')
    def list_templates():
        return {'templates': list(PROMPT_TEMPLATES.keys())}

    @app.post('/chat', response_model=ChatResponse)
    def chat(req: ChatRequest):
        try:
            return ChatResponse(reply=run_model(model, req.message, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    @app.post('/render/{name}', response_model=ChatResponse)
    def render_chat(name: str, req: ChatRequest):
        if name not in PROMPT_TEMPLATES:
            raise HTTPException(status_code=404, detail=f'template {name!r} not found')
        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)
        try:
            return ChatResponse(reply=run_model(model, prompt, req.temperature),
                                model=model)
        except Exception as e:
            raise HTTPException(status_code=503, detail=f'Model unavailable: {e}')

    return app


def check_health(client) -> bool:
    """Ping the backend's GET /health through an injected HTTP client.

    Returns True only if the request succeeds with 200 AND status == 'ok'.
    Any exception (backend down, connection refused) -> False, never raises.
    The `client` is duck-typed: an httpx.Client in production, a TestClient in
    tests — both expose .get / .post / .request.
    """
    try:
        resp = client.get('/health')
        return resp.status_code == 200 and resp.json().get('status') == 'ok'
    except Exception:
        return False


def post_chat(client, message: str, temperature: float = 0.7) -> dict:
    """POST /chat honouring the JSON contract {message, temperature}.

    Returns the parsed {reply, model} on 200. On a non-200 status returns
    {'error': ..., 'status': code}; on a connection failure returns
    {'error': ...}. The frontend never sees a raw exception.
    """
    try:
        resp = client.post('/chat', json={'message': message, 'temperature': temperature})
    except Exception as e:
        return {'error': f'request failed: {e}'}
    if resp.status_code != 200:
        return {'error': f'backend returned {resp.status_code}', 'status': resp.status_code}
    return resp.json()


def request_json(client, method: str, path: str, payload: dict = None) -> dict:
    """Call the backend and normalise EVERY outcome into one envelope:

        {'ok': bool, 'status': int | None, 'data': dict | None, 'error': str | None}

    - success (2xx):        ok=True,  status=code, data=json
    - error status (4xx/5xx): ok=False, status=code, error='HTTP <code>'
    - connection failure:   ok=False, status=None, error='connection error: ...'

    One shape for the whole frontend to branch on — no scattered try/except.
    """
    try:
        resp = client.request(method, path, json=payload)
    except Exception as e:
        return {'ok': False, 'status': None, 'data': None,
                'error': f'connection error: {e}'}
    ok = 200 <= resp.status_code < 300
    try:
        data = resp.json()
    except Exception:
        data = None
    return {
        'ok':     ok,
        'status': resp.status_code,
        'data':   data if ok else None,
        'error':  None if ok else f'HTTP {resp.status_code}',
    }


def add_cors(app: FastAPI, origins: list) -> FastAPI:
    """Enable CORS so a browser front-end on a DIFFERENT origin can call this API.

    A browser enforces the same-origin policy: JavaScript on http://localhost:8501
    may not call http://localhost:8000 unless the server opts in with CORS
    headers. CORSMiddleware adds the `access-control-allow-origin` header (and
    answers preflight OPTIONS requests) for the origins you allow.
    """
    app.add_middleware(
        CORSMiddleware,
        allow_origins=origins,
        allow_credentials=True,
        allow_methods=['*'],
        allow_headers=['*'],
    )
    return app


class AIAppClient:
    """The frontend's typed gateway to the AI backend.

    Wraps an injected HTTP client (httpx.Client in production, TestClient in
    tests) so the exact same code runs against a live server or in-process.
    Every method returns plain data the UI can render — no HTTP details leak out.
    """

    def __init__(self, client):
        self.client = client

    def health(self) -> bool:
        return check_health(self.client)

    def chat(self, message: str, temperature: float = 0.7) -> dict:
        return post_chat(self.client, message, temperature)

    def templates(self) -> list:
        env = request_json(self.client, 'GET', '/templates')
        return env['data']['templates'] if env['ok'] else []

    def render(self, name: str, topic: str, temperature: float = 0.7) -> dict:
        env = request_json(self.client, 'POST', f'/render/{name}',
                           {'message': topic, 'temperature': temperature})
        return env['data'] if env['ok'] else {'error': env['error']}

## Provided: Full-Stack File Writer

In [ ]:
from pathlib import Path

# The two runnable files of the full-stack app, embedded as strings so the
# notebook can write them verbatim (no inspect.getsource under nbconvert).
_BACKEND_SRC = 'import warnings\nwarnings.filterwarnings(\'ignore\')\nfrom fastapi import FastAPI, HTTPException\nfrom fastapi.middleware.cors import CORSMiddleware\nfrom pydantic import BaseModel, Field\nimport ollama\n\n\nclass ChatRequest(BaseModel):\n    message: str = Field(min_length=1, description=\'User message for the model\')\n    temperature: float = Field(default=0.7, ge=0.0, le=1.0)\n\n\nclass ChatResponse(BaseModel):\n    reply: str\n    model: str\n\n\nclass HealthResponse(BaseModel):\n    status: str\n    model: str\n\n\nPROMPT_TEMPLATES = {\n    \'summary\':  \'Summarize the following topic in two sentences: {topic}\',\n    \'explain\':  \'Explain {topic} to a complete beginner.\',\n    \'critique\': \'List three criticisms of {topic}.\',\n}\n\n\ndef run_model(model: str, prompt: str, temperature: float = 0.7) -> str:\n    resp = ollama.chat(\n        model=model,\n        messages=[{\'role\': \'user\', \'content\': prompt}],\n        options={\'temperature\': temperature},\n    )\n    return resp[\'message\'][\'content\'].strip()\n\n\ndef build_api(model: str = \'llama3.2\') -> FastAPI:\n    app = FastAPI(title=\'AI API\', version=\'1.0.0\')\n\n    @app.get(\'/health\', response_model=HealthResponse)\n    def health():\n        return HealthResponse(status=\'ok\', model=model)\n\n    @app.get(\'/templates\')\n    def list_templates():\n        return {\'templates\': list(PROMPT_TEMPLATES.keys())}\n\n    @app.post(\'/chat\', response_model=ChatResponse)\n    def chat(req: ChatRequest):\n        try:\n            return ChatResponse(reply=run_model(model, req.message, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    @app.post(\'/render/{name}\', response_model=ChatResponse)\n    def render_chat(name: str, req: ChatRequest):\n        if name not in PROMPT_TEMPLATES:\n            raise HTTPException(status_code=404, detail=f\'template {name!r} not found\')\n        prompt = PROMPT_TEMPLATES[name].format(topic=req.message)\n        try:\n            return ChatResponse(reply=run_model(model, prompt, req.temperature),\n                                model=model)\n        except Exception as e:\n            raise HTTPException(status_code=503, detail=f\'Model unavailable: {e}\')\n\n    return app\n\n\ndef add_cors(app: FastAPI, origins: list) -> FastAPI:\n    """Enable CORS so a browser front-end on a DIFFERENT origin can call this API.\n\n    A browser enforces the same-origin policy: JavaScript on http://localhost:8501\n    may not call http://localhost:8000 unless the server opts in with CORS\n    headers. CORSMiddleware adds the `access-control-allow-origin` header (and\n    answers preflight OPTIONS requests) for the origins you allow.\n    """\n    app.add_middleware(\n        CORSMiddleware,\n        allow_origins=origins,\n        allow_credentials=True,\n        allow_methods=[\'*\'],\n        allow_headers=[\'*\'],\n    )\n    return app\n\n\napp = build_api()\nadd_cors(app, [\'http://localhost:8501\'])\n\n\nif __name__ == \'__main__\':\n    import uvicorn\n    uvicorn.run(app, host=\'0.0.0.0\', port=8000)\n'
_FRONTEND_SRC = 'import streamlit as st\nimport httpx\n\n\ndef check_health(client) -> bool:\n    """Ping the backend\'s GET /health through an injected HTTP client.\n\n    Returns True only if the request succeeds with 200 AND status == \'ok\'.\n    Any exception (backend down, connection refused) -> False, never raises.\n    The `client` is duck-typed: an httpx.Client in production, a TestClient in\n    tests — both expose .get / .post / .request.\n    """\n    try:\n        resp = client.get(\'/health\')\n        return resp.status_code == 200 and resp.json().get(\'status\') == \'ok\'\n    except Exception:\n        return False\n\n\ndef post_chat(client, message: str, temperature: float = 0.7) -> dict:\n    """POST /chat honouring the JSON contract {message, temperature}.\n\n    Returns the parsed {reply, model} on 200. On a non-200 status returns\n    {\'error\': ..., \'status\': code}; on a connection failure returns\n    {\'error\': ...}. The frontend never sees a raw exception.\n    """\n    try:\n        resp = client.post(\'/chat\', json={\'message\': message, \'temperature\': temperature})\n    except Exception as e:\n        return {\'error\': f\'request failed: {e}\'}\n    if resp.status_code != 200:\n        return {\'error\': f\'backend returned {resp.status_code}\', \'status\': resp.status_code}\n    return resp.json()\n\n\ndef request_json(client, method: str, path: str, payload: dict = None) -> dict:\n    """Call the backend and normalise EVERY outcome into one envelope:\n\n        {\'ok\': bool, \'status\': int | None, \'data\': dict | None, \'error\': str | None}\n\n    - success (2xx):        ok=True,  status=code, data=json\n    - error status (4xx/5xx): ok=False, status=code, error=\'HTTP <code>\'\n    - connection failure:   ok=False, status=None, error=\'connection error: ...\'\n\n    One shape for the whole frontend to branch on — no scattered try/except.\n    """\n    try:\n        resp = client.request(method, path, json=payload)\n    except Exception as e:\n        return {\'ok\': False, \'status\': None, \'data\': None,\n                \'error\': f\'connection error: {e}\'}\n    ok = 200 <= resp.status_code < 300\n    try:\n        data = resp.json()\n    except Exception:\n        data = None\n    return {\n        \'ok\':     ok,\n        \'status\': resp.status_code,\n        \'data\':   data if ok else None,\n        \'error\':  None if ok else f\'HTTP {resp.status_code}\',\n    }\n\n\nclass AIAppClient:\n    """The frontend\'s typed gateway to the AI backend.\n\n    Wraps an injected HTTP client (httpx.Client in production, TestClient in\n    tests) so the exact same code runs against a live server or in-process.\n    Every method returns plain data the UI can render — no HTTP details leak out.\n    """\n\n    def __init__(self, client):\n        self.client = client\n\n    def health(self) -> bool:\n        return check_health(self.client)\n\n    def chat(self, message: str, temperature: float = 0.7) -> dict:\n        return post_chat(self.client, message, temperature)\n\n    def templates(self) -> list:\n        env = request_json(self.client, \'GET\', \'/templates\')\n        return env[\'data\'][\'templates\'] if env[\'ok\'] else []\n\n    def render(self, name: str, topic: str, temperature: float = 0.7) -> dict:\n        env = request_json(self.client, \'POST\', f\'/render/{name}\',\n                           {\'message\': topic, \'temperature\': temperature})\n        return env[\'data\'] if env[\'ok\'] else {\'error\': env[\'error\']}\n\n\nBACKEND_URL = \'http://localhost:8000\'\n\nst.set_page_config(page_title=\'Full-Stack AI Chat\', page_icon=\'🔗\')\nst.title(\'🔗 Full-Stack AI Chat\')\n\n\n@st.cache_resource\ndef get_client():\n    """One HTTP client + gateway per session (cached across reruns)."""\n    return AIAppClient(httpx.Client(base_url=BACKEND_URL, timeout=60.0))\n\n\napi = get_client()\n\n# Health badge: does the backend answer?\nif api.health():\n    st.success(\'Backend online\')\nelse:\n    st.error(\'Backend offline \\u2014 start it with:  uvicorn backend:app --reload\')\n\nif \'messages\' not in st.session_state:\n    st.session_state.messages = []\n\nfor m in st.session_state.messages:\n    with st.chat_message(m[\'role\']):\n        st.markdown(m[\'content\'])\n\nprompt = st.chat_input(\'Type a message...\')\nif prompt:\n    st.session_state.messages.append({\'role\': \'user\', \'content\': prompt})\n    with st.spinner(\'Calling backend...\'):\n        result = api.chat(prompt)\n    reply = result.get(\'reply\') or result.get(\'error\') or \'(no response)\'\n    st.session_state.messages.append({\'role\': \'assistant\', \'content\': reply})\n    st.rerun()\n'


def write_full_stack(directory: str = '.') -> tuple:
    """Write backend.py + frontend.py into `directory`; return (backend, frontend) paths."""
    d = Path(directory)
    d.mkdir(parents=True, exist_ok=True)
    (d / 'backend.py').write_text(_BACKEND_SRC, encoding='utf-8')
    (d / 'frontend.py').write_text(_FRONTEND_SRC, encoding='utf-8')
    return str(d / 'backend.py'), str(d / 'frontend.py')

## Your Pipeline

In [ ]:
# TODO: backend = TestClient(build_api())
# TODO: api = AIAppClient(backend)
# TODO: print('health   :', api.health())
# TODO: print('chat     :', api.chat('Say hello in 3 words.'))
# TODO: print('templates:', api.templates())
# TODO: print('render   :', api.render('summary', 'FastAPI'))
#
# TODO: b, f = write_full_stack('.')
# TODO: print('Wrote', b, 'and', f)
# TODO: print('Run:  uvicorn backend:app --reload   (terminal 1)')
# TODO: print('Run:  streamlit run frontend.py       (terminal 2)')

## Checks

In [ ]:
import os


def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: AIAppClient works against the in-process backend
    try:
        assert 'api' in globals() and isinstance(api, AIAppClient), 'create api = AIAppClient(...)'
        assert api.health() is True, 'api.health() should be True'
        passed += 1; print('✅ Check 1: AIAppClient talks to the backend')
    except Exception as e:
        print(f'❌ Check 1: {e}')
        print(f'\nScore: {passed}/{total}'); return

    # Check 2: both files written
    try:
        assert os.path.exists('backend.py'), 'backend.py not found — call write_full_stack()'
        assert os.path.exists('frontend.py'), 'frontend.py not found — call write_full_stack()'
        passed += 1; print('✅ Check 2: backend.py + frontend.py written')
    except Exception as e:
        print(f'❌ Check 2: {e}')
        print(f'\nScore: {passed}/{total}'); return

    back = open('backend.py', encoding='utf-8').read()
    front = open('frontend.py', encoding='utf-8').read()

    # Check 3: backend.py is a FastAPI app with CORS + module-level app
    try:
        assert 'from fastapi import FastAPI' in back
        assert 'CORSMiddleware' in back, 'backend.py must enable CORS'
        assert 'app = build_api()' in back, 'backend.py must expose module-level app'
        passed += 1; print('✅ Check 3: backend.py has FastAPI + CORS + app')
    except Exception as e:
        print(f'❌ Check 3: {e}')

    # Check 4: frontend.py is a Streamlit app using httpx + AIAppClient
    try:
        assert 'import streamlit as st' in front
        assert 'httpx.Client' in front, 'frontend.py must use httpx.Client'
        assert 'AIAppClient' in front, 'frontend.py must use AIAppClient'
        assert 'st.chat_input' in front, 'frontend.py must have a chat UI'
        passed += 1; print('✅ Check 4: frontend.py wires httpx -> AIAppClient -> UI')
    except Exception as e:
        print(f'❌ Check 4: {e}')

    # Check 5: both files compile
    try:
        compile(back, 'backend.py', 'exec')
        compile(front, 'frontend.py', 'exec')
        passed += 1; print('✅ Check 5: both files compile as valid Python')
    except Exception as e:
        print(f'❌ Check 5: {e}')

    if passed == total:
        print('🎉 Project complete! Two terminals: uvicorn backend:app  +  streamlit run frontend.py')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()